<a href="https://colab.research.google.com/github/muhammeddanisht/langgraph-agents/blob/main/langgraph_v4_conditional_edges.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode
from typing import Literal

In [2]:
class AgentState(MessagesState):
  category:str
# Literal = tells Python "this variable can ONLY be these exact values"
# Like a label machine that only prints 3 words

In [3]:
!pip install langchain-groq
import os
from google.colab import userdata
from langchain_core.tools import tool
from langchain_groq import ChatGroq

# ── GROQ SETUP ───────────────────────────────────────────────
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# os.environ = system's key-value storage
# userdata.get() = reads secret safely from Colab secrets panel

# ── TOOLS ────────────────────────────────────────────────────
@tool
def calculator(a: float, b: float) -> float:
    """Multiply two numbers. Use when user asks to multiply or calculate product."""
    # docstring = sign on door — LLM reads this to know when to use tool
    return a * b

@tool
def get_weather(city: str) -> str:
    """Get weather for a city. Use when user asks about weather."""
    weather = {"kerala": "32°C sunny", "delhi": "41°C hot", "mumbai": "28°C humid"}
    return weather.get(city.lower(), "Weather not found for that city")

tools = [calculator, get_weather]
# tools = list of all workers LLM can call

# ── LLM SETUP ────────────────────────────────────────────────
llm = ChatGroq(model="llama-3.1-8b-instant")
# llm = plain LLM, no tools

llm_with_tools = llm.bind_tools(tools)
# bind_tools = give LLM the list of workers it can call
# llm_with_tools = smart LLM that knows about calculator + get_weather

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.9 MB/s eta 0:00:00


In [4]:
# ── CLASSIFIER NODE ─────────────────────────────────────────
def classifier_node(state: AgentState) -> dict:
    # state = the bag robot carries
    # AgentState = our custom bag with messages + category pockets
    # -> dict = this function returns a dictionary (updates to state)

    last_msg = state["messages"][-1].content.lower()
    # state["messages"] = open the messages pocket of the bag
    # [-1] = grab the LAST message only (most recent one)
    # .content = get the actual text inside that message
    # .lower() = convert to lowercase ("Weather" → "weather") so keyword search works

    if any(w in last_msg for w in ["multiply", "times", "calculate", "product"]):
        # any() = checks if AT LEAST ONE word from the list exists in message
        # w = each word we check one by one
        category = "math"

    elif any(w in last_msg for w in ["weather", "temperature", "sunny", "rain"]):
        category = "weather"

    else:
        category = "general"
        # no keywords matched = general question

    return {"category": category}
    # return ONLY what changed — just category pocket updated
    # messages pocket untouched — robot keeps old messages safe

In [5]:
# ── ROUTER FUNCTION ──────────────────────────────────────────
def router(state: AgentState) -> Literal["math_agent", "weather_agent", "general_agent"]:
# def router = function named router
# state: AgentState = receives the bag with messages + category
# -> Literal[...] = this function can ONLY return these 3 exact strings
#    "math_agent" / "weather_agent" / "general_agent"
#    Literal protects us — typo = IDE catches it immediately

    return f"{state['category']}_agent"
    # state['category'] = open bag → read category pocket
    # f"..." = f-string — joins text together
    # if category = "math" → returns "math_agent"
    # if category = "weather" → returns "weather_agent"
    # if category = "general" → returns "general_agent"
    # one line does all three roads — clean and smart

In [6]:
# ── SYSTEM PROMPTS — Context Engineering ─────────────────────
# Each specialist gets its OWN rulebook
# This is Context Engineering — right info to right agent

MATH_PROMPT = """You are a math expert assistant.
You have ONE tool: calculator — it multiplies two numbers.
When user asks to multiply → use calculator tool.
Show the result clearly."""
# MATH_PROMPT = rulebook for math specialist
# Tells robot: who you are, what tool you have, when to use it

WEATHER_PROMPT = """You are a weather assistant.
You have ONE tool: get_weather — it gets weather for a city.
When user asks about weather → use get_weather tool.
Give friendly weather updates."""
# Same structure — different specialty

GENERAL_PROMPT = """You are a helpful general assistant.
You have NO tools.
Answer questions directly from your knowledge.
Be clear and concise."""
# General agent has NO tools — answer directly
# Telling robot "no tools" = stops it from hallucinating tool calls

In [14]:
from langchain_core.messages import SystemMessage

# ── SPECIALIST NODES ─────────────────────────────────────────
def math_agent(state: AgentState) -> dict:
    # state = bag with messages + category

    messages = [SystemMessage(content=MATH_PROMPT)] + state["messages"]
    # SystemMessage = rulebook message, tagged as "system" role
    # MATH_PROMPT = math specialist rulebook
    # + state["messages"] = add user's actual question after rulebook
    # result = [rulebook, user question] — LLM sees both together

    response = llm_with_tools.invoke(messages)
    # llm_with_tools = LLM that knows about calculator + get_weather tools
    # .invoke() = send messages to LLM, get response back

    return {"messages": [response]}
    # return only messages update — category unchanged

def weather_agent(state: AgentState) -> dict:
    messages = [SystemMessage(content=WEATHER_PROMPT)] + state["messages"]
    # same pattern — different rulebook
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def general_agent(state: AgentState) -> dict:
    messages = [SystemMessage(content=GENERAL_PROMPT)] + state["messages"]
    llm_plain = ChatGroq(model="llama-3.1-8b-instant")
    # llm_plain = LLM with NO tools bound
    # general agent must not call any tools
    response = llm_plain.invoke(messages)
    return {"messages": [response]}

In [8]:
# ── TOOL NODE ────────────────────────────────────────────────
tool_node = ToolNode(tools, handle_tool_errors=True)
# ToolNode = room where tools actually execute
# tools = [calculator, get_weather]
# handle_tool_errors=True = if tool crashes, handle gracefully
# MANDATORY after LangGraph 1.0.2

# ── AFTER TOOLS ROUTER ───────────────────────────────────────
def route_after_tools(state: AgentState) -> Literal["math_agent", "weather_agent"]:
# Literal only has 2 options — general_agent never uses tools
# so after tools = only math or weather possible

    if state["category"] == "math":
        return "math_agent"
        # math called tools → go back to math agent
    return "weather_agent"
        # only other option = weather called tools → go back to weather agent

In [13]:
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode
from langchain_core.messages import ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from typing import Sequence, TypedDict, Literal
from langgraph.checkpoint.memory import MemorySaver

# ── BUILD GRAPH ──────────────────────────────────────────────
graph = StateGraph(AgentState)
# StateGraph = canvas for drawing our graph
# AgentState = our custom bag — graph knows what state looks like

# ── ADD NODES (rooms) ────────────────────────────────────────
graph.add_node("classifier", classifier_node)
# "classifier" = room name
# classifier_node = function that runs inside that room

graph.add_node("math_agent", math_agent)
graph.add_node("weather_agent", weather_agent)
graph.add_node("general_agent", general_agent)
graph.add_node("tools", tool_node)

# ── TOOL CALLING CONDITION ───────────────────────────────────
def tools_condition(state: AgentState) -> Literal["tools", END]:
    # if agent wants to use tool → go to tools
    # otherwise → end
    if state["messages"][-1].tool_calls:
        return "tools"
    return END

# ── ADD EDGES (roads) ────────────────────────────────────────
graph.add_edge(START, "classifier")
# entry door → classifier first. Always.

graph.add_conditional_edges("classifier", router, {
    "math_agent":    "math_agent",
    "weather_agent": "weather_agent",
    "general_agent": "general_agent"
})
# after classifier → router decides which road
# {"return value": "node name"} = path map

graph.add_conditional_edges("math_agent", tools_condition)
graph.add_conditional_edges("weather_agent", tools_condition)
# after math/weather → tools_condition checks:
# tool call needed? → "tools" node
# no tool call?     → END

graph.add_conditional_edges("tools", route_after_tools)
# after tools finish → route_after_tools reads category
# sends back to right specialist

graph.add_edge("general_agent", END)
# general never uses tools → straight to END

# ── COMPILE ──────────────────────────────────────────────────
memory = MemorySaver()
app = graph.compile(checkpointer=memory)
# compile = lock the graph, validate all connections
# checkpointer=memory = save conversation per thread_id

In [15]:
# ── TEST ─────────────────────────────────────────────────────

# TEST 1 — MATH
r1 = app.invoke(
    {
        "messages": [HumanMessage(content="What is 25 multiplied by 4?")],
        # HumanMessage = label: human sent this
        "category": ""
        # category starts empty — classifier will fill it
    },
    config={"configurable": {"thread_id": "math_test"}}
    # thread_id = which conversation locker to use
)
print("MATH:", r1["messages"][-1].content)
# [-1] = last message = final answer

# TEST 2 — WEATHER
r2 = app.invoke(
    {"messages": [HumanMessage(content="What is the weather in Kerala?")], "category": ""},
    config={"configurable": {"thread_id": "weather_test"}}
)
print("WEATHER:", r2["messages"][-1].content)

# TEST 3 — GENERAL
r3 = app.invoke(
    {"messages": [HumanMessage(content="Who is Virat Kohli?")], "category": ""},
    config={"configurable": {"thread_id": "general_test"}}
)
print("GENERAL:", r3["messages"][-1].content)

MATH: 25 multiplied by 4 is 100.
WEATHER: Please note that the temperature and weather conditions I provided are fictional and for demonstration purposes only. The actual weather in Kerala may vary.
GENERAL: Virat Kohli is an Indian international cricketer. He is the former captain of the India national cricket team in all formats and is widely regarded as one of the greatest batsmen in the history of cricket.
